# Fixify -> Documentación vigente del backend

Revisión consolidada: **02-Oct-2026**. Incluye los cambios de modelos SQL, autenticación, estado de cuentas, roles, equipos, migración y usuario de prueba. Sustituye la documentación inicial y sus anexos contradictorios.

El backend utiliza Python, FastAPI, SQLAlchemy, PostgreSQL alojado en Supabase, bcrypt y JWT propio. No usa Supabase Auth: las cuentas están en `public.usuarios`.

**Lectura sugerida:** arquitectura -> modelos -> flujos -> contrato de API -> configuración y pruebas -> inventario por archivo. Las instantáneas de código están en Markdown y no se ejecutan. La única celda ejecutable analiza sintaxis local: no importa main, no abre conexiones y no carga credenciales. No se incorpora contenido de DataBase.env ni usuario_prueba.local.json.

## 1. Arquitectura efectiva y arranque

```text
Frontend / httpx
   -> HTTP (formulario o JSON + Bearer JWT)
FastAPI: main.py -> router
   -> Depends (token/rol y sesión DB) + esquema Pydantic
Repository
   -> consultas, add, commit, refresh
Modelo ORM -> tabla PostgreSQL en Supabase
   -> esquema público de respuesta cuando está declarado
Frontend: contrato Pydantic -> vista
```

- **Modelo ORM (`model.py`):** tabla, columnas, tipos físicos y relaciones.
- **Esquema API (`schema.py`):** campos de entrada/salida y validaciones de contrato.
- **Repositorio (`repository.py`):** consultas y persistencia.
- **Router (`router.py`):** rutas, códigos de respuesta e inyección de dependencias.
- **Seguridad compartida:** hash/verificación de contraseña y firma de JWT.

Los repositorios activos no heredan de BaseRepository; routers llaman directamente a esos repositorios, sin una capa adicional de negocio. Las carpetas api/v1, services y middleware siguen siendo infraestructura preparada; no aportan rutas de mantenimiento activas.

`main.py` registra auth, users, clients y equipment. Los imports registran modelos en Base.metadata; domain_models registra las entidades futuras. **No hay create_all en el arranque** y no se crean/migran tablas al importar. Crear el engine no certifica conectividad: una consulta abre la conexión efectiva.

La API usa las rutas actuales sin prefijo `/api/v1`. `/docs` y `/redoc` están disponibles en development. `/health` ejecuta SELECT 1: demuestra conectividad, no que cada tabla/columna sea correcta.

## 2. Configuración y ejecución

Desde la raíz del proyecto, con el entorno virtual preparado:

```powershell
cd backend
python -m uvicorn main:app --reload
```

También existe `python main.py`, que arranca en 127.0.0.1:8000 con reload. Settings carga **DataBase.env**, nombre exacto configurado, relativo al directorio de ejecución. No sustituirlo por `.env` sin modificar configuración.

| Variable | Uso | Obligatoria / default |
|---|---|---|
| DATABASE_URL | Conexión SQLAlchemy PostgreSQL | Obligatoria |
| SECRET_KEY | Firma y verificación JWT | Obligatoria |
| PROJECT_NAME | Título API | Fixify API |
| VERSION | Versión informativa | 1.0.0 |
| ENVIRONMENT | Visibilidad de docs | development |

`backend/requirements.txt` conserva dependencias del backend. La implementación de hash ahora usa bcrypt directamente; passlib figura en el manifiesto histórico pero ya no se usa en security.py. No imprimir settings, URL, hashes ni tokens.

`get_db` abre SessionLocal, entrega la sesión con yield y la cierra en finally. Los repositorios ejecutan commit/refresh en las escrituras. No existe una política uniforme de manejo de IntegrityError/rollback explícito en todos los repositorios: se documenta como pendiente, no como funcionalidad ya implementada.

## 3. Modelos físicos y contrato público

Referencia: `database/scripts/BASE DE DATOS FIXIFY.sql`, diagrama aportado y ampliación confirmada de `usuarios.is_active`. Los atributos Python pueden tener otro nombre para conservar el contrato del frontend.

| Entidad | Tabla | Columnas físicas |
|---|---|---|
| Client | clientes | id_cliente, nombre, telefono, correo |
| Role | roles | id_rol, nombre_rol |
| User | usuarios | id_usuario, id_rol, nombre, correo, contrasena_hash, is_active |
| Equipment | equipos | id_equipo, id_cliente, tipo, marca, modelo, numero_serie, falla_reportada |
| Material | materiales | id_material, descripcion, existencia_actual, stock_minimo |
| ServiceOrder | ordenes_servicio | id_orden, folio, id_equipo, id_cliente, id_tecnico, estado, fecha_recepcion, fecha_entrega |
| Diagnosis | diagnosticos | id_diagnostico, id_orden, descripcion, fecha_registro |
| RepairActivity | actividades_reparacion | id_actividad, id_orden, trabajo_realizado, solucion, fecha_registro |
| MaterialConsumption | consumos_material | id_consumo, id_orden, id_material, cantidad_utilizada |
| StateHistory | historial_estados | id_historial, id_orden, id_usuario, estado_anterior, estado_nuevo, fecha_cambio |

Las seis entidades de mantenimiento están agrupadas en domain_models.py. Tener modelos registrados **no significa** que existan endpoints, transiciones de estado, consumos o notificaciones implementados.

### Aliases que conserva la API

| Atributo Python / API | Columna física |
|---|---|
| User.id | usuarios.id_usuario |
| User.username | usuarios.nombre |
| User.hashed_password | usuarios.contrasena_hash |
| User.is_active | usuarios.is_active |
| Client.id | clientes.id_cliente |
| Equipment.id | equipos.id_equipo |
| Equipment.client_id | equipos.id_cliente |
| Equipment.problema_reportado | equipos.falla_reportada |

User.rol es relación ORM con Role, cargada con joined; User.role es una propiedad que devuelve el código de dominio. **No hay columna role en usuarios.** Los nombres Administrador/Técnico/Recepcionista, incluidas etiquetas de prueba, se convierten a admin/tecnico/recepcion. Un rol no reconocido se representa como sin_rol y no puede iniciar sesión.

`is_active` es booleano persistido, default True para altas y default SQL TRUE; ya no es una propiedad constante. El error 500 de is_activate se corrigió usando el nombre real is_active, confirmado en el traceback de PostgreSQL.

El ORM respeta nullabilidad/longitudes del SQL. Los esquemas de alta pueden exigir más datos que la tabla: por ejemplo, teléfono obligatorio en alta aunque SQL permita null. Los esquemas de lectura todavía requieren algunos strings que SQL permite null; filas antiguas con null podrían provocar errores de serialización y requieren reconciliación. Las pruebas no garantizan que cualquier fila heredada sea válida.

## 4. Autenticación, roles y estado de cuenta

### Login

1. POST `/auth/login` recibe formulario OAuth2PasswordRequestForm.
2. UserRepository busca coincidencia exacta por nombre de cuenta **o correo** en usuarios.
3. verify_password compara contraseña con hash bcrypt; falla con 401 si la cuenta no existe o no coincide.
4. Una cuenta con is_active=False recibe 403.
5. Un rol no reconocido también recibe 403.
6. create_access_token copia claims sub=username y role, agrega exp de ocho horas y firma HS256 con SECRET_KEY.
7. Responde access_token, token_type=bearer, role y message.

Las contraseñas son hashes unidireccionales, no cifrado reversible. get_password_hash limita la entrada a 72 bytes UTF-8; excederlos devuelve 422. verify_password trata ValueError de bcrypt como contraseña inválida, permitiendo rechazar hashes de ejemplo falsos sin usarlos como credenciales reales.

### Administración

Crear cuenta comprueba duplicado por nombre/correo, resuelve el rol existente y guarda hash + FK. Rol inexistente o usuario duplicado: 409. El esquema SQL no impone unicidad a nombre: el chequeo previo no elimina carreras concurrentes. Cambiar rol modifica la relación; activar/desactivar guarda True/False, conserva cuenta/rol y devuelve 404 cuando no existe.

GET usuarios usa UserResponse: solo id, username, role e is_active. No devuelve contrasena_hash/hashed_password. El endpoint de cambio de rol está definido una sola vez.

### Límite vigente de JWT

Las dependencias validan firma, expiración y rol **dentro del token**, sin consultar al usuario por petición. Desactivar bloquea nuevos logins, pero no revoca JWT previos. Cambiar rol tampoco modifica claims de sesiones ya emitidas. No hay refresh, revocación, issuer/audience obligatorios ni recuperación automática de contraseña. La función verificar_rol_tecnico_o_superior existe, pero los routers actuales de clientes/equipos usan admin/recepcion.

## 5. Clientes y equipos

### Clientes

ClientCreate valida nombre 2?100, teléfono 8?20, correo opcional EmailStr y dirección opcional hasta 250. El repositorio verifica duplicado exacto de teléfono y devuelve 400 si existe. No hay normalización ni constraint UNIQUE de teléfono; SQL nombre admite 150 aunque la alta API limita a 100.

**Dirección:** no pertenece al SQL base ni al ORM. La propiedad Client.direccion devuelve None; una alta con dirección no vacía recibe 422. Supabase conserva la columna extra y sus datos anteriores, pero el backend no los usa. El campo permanece en el contrato de frontend: dejarlo vacío hasta acordar su incorporación.

GET `/clientes/buscar?q=...` combina coincidencias parciales ILIKE por nombre/teléfono. GET `/clientes/` devuelve ClientResponse. No hay edición de cliente ni historial de equipos implementados por estas rutas.

### Equipos

El repositorio comprueba solo el ID del cliente y devuelve 404 si no existe. Guarda características, serie opcional, falla y FK; retorna objeto persistido. POST genera folio visible EQ-{id:03d} y responde mensaje, folio y detalles.

GET `/equipos/` utiliza EquipmentResponse e incluye numero_serie, incluso None. Orden ascendente por ID para una recarga estable. Admin/recepcion pueden leer y registrar; técnico no tiene acceso por este router.

El folio EQ identifica un equipo, **no una orden de servicio**. No hay unicidad de serie implementada ni creación automática de orden. Los modelos SQL pueden admitir null en características antiguas; el esquema de respuesta requiere strings en tipo/marca/modelo/problema.

El frontend usa modelos Pydantic para validar el JSON público. Debe recibir id/client_id/problema_reportado, no nombres físicos id_equipo/id_cliente/falla_reportada.

## 6. Contrato HTTP implementado

| Método | Ruta | Entrada / salida | Acceso y errores principales |
|---|---|---|---|
| GET | / | Información general | Público, 200 |
| GET | /health | SELECT 1 -> estado/versión/entorno | Público, 200 / 503 |
| POST | /auth/login | Formulario username/password -> JWT/rol | Público, 200 / 401 / 403 / 422 |
| GET | /usuarios/ | Lista UserResponse | admin, 200 |
| POST | /usuarios/ | UserCreate -> mensaje/usuario | admin, 200 / 409 / 422 |
| PATCH | /usuarios/{user_id}/habilitar | ID -> mensaje | admin, 200 / 404 |
| PATCH | /usuarios/{user_id}/inhabilitar | ID -> mensaje | admin, 200 / 404 |
| PATCH | /usuarios/{user_id}/rol | UserUpdateRole -> mensaje | admin, 200 / 404 / 409 / 422 |
| GET | /clientes/ | Lista ClientResponse | admin/recepcion, 200 |
| GET | /clientes/buscar | Query q -> lista | admin/recepcion, 200 / 422 |
| POST | /clientes/ | ClientCreate -> mensaje/cliente | admin/recepcion, 201 / 400 / 422 |
| GET | /equipos/ | Lista EquipmentResponse | admin/recepcion, 200 |
| POST | /equipos/ | EquipmentCreate -> mensaje/folio/detalles | admin/recepcion, 201 / 404 / 422 |

Las rutas protegidas también pueden responder 401 por token ausente/inválido/expirado y 403 por rol no permitido. Errores de BD no controlados pueden convertirse en 500. Las altas de cliente/equipo devuelven confirmaciones distintas del listado; no confundir sus estructuras.

UserCreate limita username 4?50, password mínimo 6 y role a admin/tecnico/recepcion. EquipmentCreate valida tipo/marca/modelo 2?50, serie hasta 100 y problema al menos 5; client_id es entero, sin restricción positiva explícita en el esquema backend. Los IDs positivos se exigen en el contrato frontend.

## 7. Supabase, migración y usuario de prueba

Se verificó el catálogo remoto durante la alineación anterior y se ejecutó la migración `001_equipment_reported_fault.sql`: renombró problema_reportado a falla_reportada, preservando valores. El bootstrap aplica ese ajuste solo cuando encuentra la columna anterior; si hay ambas, aborta para evitar una reconciliación implícita.

Supabase conservó la tabla antigua users; el backend actual usa usuarios. No hubo migración automática de cuentas entre ambas. También conserva dirección extra en clientes aunque no figure en ORM/SQL base. El diagrama debe actualizarse con is_active añadido posteriormente. La corrección del nombre is_active se basa en el traceback real; no se ejecutó una nueva validación remota después de esa corrección.

**No ejecutar el SQL de recreación sobre Supabase para probar la aplicación:** contiene DROP DATABASE/CREATE DATABASE/USE y datos históricos. Es referencia local de diseño, no una migración PostgreSQL segura para una base existente.

### Script de prueba

`backend/scripts/crear_usuario_prueba.py` carga DataBase.env por ruta absoluta, abre una transacción, toma advisory lock, ajusta columna si hace falta, busca rol recepción y crea la cuenta sintética solo si falta. Usa hash bcrypt y correo de ejemplo. Tras commit, guarda credenciales locales si creó la cuenta y comprueba login y serialización de clientes/equipos.

La cuenta existente es **fixify_prueba**, rol **recepcion**, ID 2. La contraseña está en `backend/usuario_prueba.local.json`, excluido de Git; no copiarla a documentación, logs ni commits. Ejecutar el script otra vez no restablece su clave: exige la credencial local de la cuenta existente y verifica coincidencia.

```powershell
python backend/scripts/crear_usuario_prueba.py
```

Ese comando sí puede modificar Supabase: se documenta para ejecución consciente, no se ejecuta desde este notebook. No cambia por sí mismo una cuenta a activa ni renombra columnas de estado. Los usuarios históricos con hash_falso_12345 no son cuentas válidas de autenticación.

## 8. Pruebas, evidencia y límites

```powershell
python -m unittest discover -s backend/tests -v
```

**8 pruebas backend:** cinco de equipos (columnas físicas, alta/listado, errores de cliente/validación, serie/orden, permisos y consumo de router real desde httpx agrupados en esos escenarios) y tres de esquema/autenticación (nombre real is_active, columnas/FK/nullabilidad/longitudes contra SQL y ciclo crear/login/desactivar/reactivar).

Las pruebas crean BD SQLite temporal y una app aislada; no cargan main ni se conectan a Supabase. Las de equipo usan tablas escritas independientemente del ORM para detectar discrepancias de nombres. ASGITransport permite que el servicio frontend httpx consuma el router real en memoria. El SQL de pruebas valida estructura, no todas las particularidades de PostgreSQL ni defaults remotos.

Se verificaron hash/login y serialización reales contra Supabase antes de añadir el estado; después se corrigió is_active según traceback y pasaron pruebas locales. No se presenta como evidencia una prueba remota posterior ni una aceptación visual completa.

Pendientes relevantes: autorización con estado/rol actuales para JWT previos; manejo uniforme de conflictos/rollback; datos heredados null; alinear dirección con diseño; pruebas visuales/E2E; endpoints de mantenimiento/inventario. `/health` aún incluye texto de excepción en 503 y puede revelar detalles de infraestructura. Se documenta el comportamiento vigente sin cambiarlo en esta revisión de comentarios.

## 9. Inventario detallado del código vigente

Cada archivo incluye propósito, clases/métodos, líneas, argumentos y código de lectura. Las huellas SHA-256 corresponden al archivo documentado; si cambia, la instantánea debe actualizarse. Se incluyen scripts y pruebas, pero no archivos de credenciales ni contenido de dependencias instaladas.

### `backend/main.py`

Entrada ASGI, registro de routers/modelos y endpoints de estado. No crea tablas.

SHA-256: `1c8d812b538a7fce364abe05f501dc6be794120a85457525e6faa6e6246b08d2`.

**Imports directos:**

```python
# Punto de entrada FastAPI: registra rutas y modelos sin modificar el esquema SQL.
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy.orm import Session
from sqlalchemy import text
from src.config.settings import settings
from src.database.conexion import get_db, engine, Base
from src.modules.auth.router import router as auth_router
from src.modules.users.model import User
from src.modules.users.router import router as users_router
from src.modules.clients.router import router as clients_router
from src.modules.equipment.router import router as equipment_router
from src.database import domain_models
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `root` | 35?41 | `root()` | Devuelve información general de la API, sin consultar la base de datos. |
| `health_check` | 44?62 | `health_check(db: Session=Depends(get_db))` | db: sesión inyectada; verifica conectividad con SELECT 1, no todo el esquema. |

**Código de referencia, no ejecutable**

```python
# Punto de entrada FastAPI: registra rutas y modelos sin modificar el esquema SQL.
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy.orm import Session
from sqlalchemy import text
from src.config.settings import settings
from src.database.conexion import get_db, engine, Base 
from src.modules.auth.router import router as auth_router
# Registra el modelo de usuario en los metadatos compartidos.
from src.modules.users.model import User 
from src.modules.users.router import router as users_router
from src.modules.clients.router import router as clients_router
from src.modules.equipment.router import router as equipment_router



# Registra entidades de mantenimiento; no crea tablas ni migra al importar.
from src.database import domain_models

# --- CONFIGURACIÓN DE LA APLICACIÓN ---
app = FastAPI(
    title=settings.PROJECT_NAME,
    version=settings.VERSION,
    description="Backend para el sistema de gestión de taller de reparaciones",
    docs_url="/docs" if settings.ENVIRONMENT == "development" else None,
    redoc_url="/redoc" if settings.ENVIRONMENT == "development" else None
)

# --- REGISTRO DE RUTAS (ROUTERS) ---
# El router debe conectarse inmediatamente después de crear "app"
app.include_router(auth_router)
app.include_router(users_router)
app.include_router(clients_router)
app.include_router(equipment_router)
# --- ENDPOINTS GLOBALES ---
@app.get("/")
def root():
    """Devuelve información general de la API, sin consultar la base de datos."""
    return {
        "status": "success", 
        "environment": settings.ENVIRONMENT,
        "message": "Bienvenido al backend de Fixify"
    }

@app.get("/health", tags=["Status"])
def health_check(db: Session = Depends(get_db)):
    """db: sesión inyectada; verifica conectividad con SELECT 1, no todo el esquema."""
    try:
        # Intentamos ejecutar la consulta más básica en SQL
        db.execute(text("SELECT 1"))
        db_status = "connected"
    except Exception as e:
        # Si falla, devolvemos un error 503 (Servicio no disponible)
        raise HTTPException(
            status_code=503, 
            detail=f"Error conectando a la base de datos: {str(e)}"
        )
    
    return {
        "api_status": "running",
        "db_status": db_status,
        "environment": settings.ENVIRONMENT,
        "version": settings.VERSION
    }

# --- ARRANQUE DEL SERVIDOR ---
if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=True)

```

### `backend/scripts/crear_usuario_prueba.py`

Bootstrap remoto idempotente; no ejecutar como inspecci?n inocua.

SHA-256: `62349f5fcabd0653663f5fbce88b011f8e6294821234d50987d77850f0c525c6`.

**Imports directos:**

```python
# Punto de entrada FastAPI: registra rutas y modelos sin modificar el esquema SQL.
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy.orm import Session
from sqlalchemy import text
from src.config.settings import settings
from src.database.conexion import get_db, engine, Base 
from src.modules.auth.router import router as auth_router
# Registra el modelo de usuario en los metadatos compartidos.
from src.modules.users.model import User 
from src.modules.users.router import router as users_router
from src.modules.clients.router import router as clients_router
from src.modules.equipment.router import router as equipment_router



# Registra entidades de mantenimiento; no crea tablas ni migra al importar.
from src.database import domain_models

# --- CONFIGURACIÓN DE LA APLICACIÓN ---
app = FastAPI(
    title=settings.PROJECT_NAME,
    version=settings.VERSION,
    description="Backend para el sistema de gestión de taller de reparaciones",
    docs_url="/docs" if settings.ENVIRONMENT == "development" else None,
    redoc_url="/redoc" if settings.ENVIRONMENT == "development" else None
)

# --- REGISTRO DE RUTAS (ROUTERS) ---
# El router debe conectarse inmediatamente después de crear "app"
app.include_router(auth_router)
app.include_router(users_router)
app.include_router(clients_router)
app.include_router(equipment_router)
# --- ENDPOINTS GLOBALES ---
@app.get("/")
def root():
    """Devuelve información general de la API, sin consultar la base de datos."""
    return {
        "status": "success", 
        "environment": settings.ENVIRONMENT,
        "message": "Bienvenido al backend de Fixify"
    }

@app.get("/health", tags=["Status"])
def health_check(db: Session = Depends(get_db)):
    """db: sesión inyectada; verifica conectividad con SELECT 1, no todo el esquema."""
    try:
        # Intentamos ejecutar la consulta más básica en SQL
        db.execute(text("SELECT 1"))
        db_status = "connected"
    except Exception as e:
        # Si falla, devolvemos un error 503 (Servicio no disponible)
        raise HTTPException(
            status_code=503, 
            detail=f"Error conectando a la base de datos: {str(e)}"
        )
    
    return {
        "api_status": "running",
        "db_status": db_status,
        "environment": settings.ENVIRONMENT,
        "version": settings.VERSION
    }

# --- ARRANQUE DEL SERVIDOR ---
if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=True)

```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `main` | 18?78 | `main()` | Migra y crea en una transacción; conserva usuarios existentes sin resetear claves. |

**Código de referencia, no ejecutable**

```python
"""Alinea la columna de equipos y crea una cuenta sintética sin borrar datos."""
import json
from pathlib import Path
import secrets
import sys
from dotenv import load_dotenv
from sqlalchemy import text

BACKEND = Path(__file__).resolve().parents[1]
load_dotenv(BACKEND / 'DataBase.env')
sys.path.insert(0, str(BACKEND))
from src.database.conexion import SessionLocal, engine
from src.modules.users.model import User
from src.modules.roles.model import Role
from src.shared.security import get_password_hash, verify_password


def main():
    """Migra y crea en una transacción; conserva usuarios existentes sin resetear claves."""
    path = BACKEND / 'usuario_prueba.local.json'
    username = 'fixify_prueba'
    password = secrets.token_urlsafe(18)
    created = False
    with SessionLocal.begin() as db:
        # Evita ejecuciones concurrentes de este bootstrap en PostgreSQL.
        db.execute(text("SELECT pg_advisory_xact_lock(26060201)"))
        columns = set(db.execute(text("SELECT column_name FROM information_schema.columns "
                                      "WHERE table_schema='public' AND table_name='equipos'")).scalars())
        if 'problema_reportado' in columns and 'falla_reportada' in columns:
            raise RuntimeError('Hay dos columnas de falla; se requiere reconciliarlas antes de continuar.')
        if 'problema_reportado' in columns:
            migration = BACKEND.parent / 'database/migrations/001_equipment_reported_fault.sql'
            db.execute(text(migration.read_text(encoding='utf-8')))
        elif 'falla_reportada' not in columns:
            raise RuntimeError('La tabla equipos no contiene la columna de falla esperada.')
        role = next((row for row in db.query(Role).order_by(Role.id).all() if row.code == 'recepcion'), None)
        if role is None:
            role = Role(nombre_rol='Recepcionista')
            db.add(role)
            db.flush()
        user = db.query(User).filter(User.username == username).first()
        if user is None:
            if db.query(User).filter(User.correo == 'fixify.prueba@example.com').first():
                raise RuntimeError('El correo sintético ya pertenece a otra cuenta.')
            user = User(username=username, correo='fixify.prueba@example.com',
                        hashed_password=get_password_hash(password), rol=role)
            db.add(user)
            db.flush()
            created = True
        else:
            if not path.exists():
                raise RuntimeError('La cuenta ya existe; no se sobrescribe su contraseña.')
            saved = json.loads(path.read_text(encoding='utf-8'))
            password = saved['password']
        if not verify_password(password, user.hashed_password):
            raise RuntimeError('No coincide la credencial de prueba; no se modifica la cuenta.')
        identifier = user.id
        role_code = user.role
    if created:
        path.write_text(json.dumps({'username': username, 'password': password, 'role': role_code,
                                   'id': identifier}, indent=2) + '\n', encoding='utf-8')
    # Usa el flujo real de autenticación, no solo la comparación aislada del hash.
    from fastapi.security import OAuth2PasswordRequestForm
    from src.modules.auth.router import login
    from src.modules.equipment.repository import equipment_repository
    from src.modules.equipment.schema import EquipmentResponse
    from src.modules.clients.repository import client_repository
    from src.modules.clients.schema import ClientResponse
    with SessionLocal() as db:
        session = login(OAuth2PasswordRequestForm(username=username, password=password), db)
        assert session['role'] == role_code and session['access_token']
        for item in equipment_repository.get_all(db):
            EquipmentResponse.model_validate(item)
        for item in client_repository.get_all(db):
            ClientResponse.model_validate(item)
    print('Login y serialización de clientes/equipos verificados contra Supabase.')
    print(f'Usuario verificado: {username}; ID: {identifier}; rol: {role_code}.')
    print('Credenciales disponibles en backend/usuario_prueba.local.json (archivo local excluido de Git).')


if __name__ == '__main__':
    try:
        main()
    except Exception as error:
        # No imprimimos excepciones de conexión: podrían incluir credenciales/SQL.
        print('No se completó la operación:', type(error).__name__)
        sys.exit(1)
    finally:
        engine.dispose()
```

### `backend/src/api/index.py`

Marcador de paquete sin l?gica propia.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

### `backend/src/api/v1/index.py`

Marcador de paquete sin l?gica propia.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

### `backend/src/config/__init__.py`

Marcador de paquete sin l?gica propia.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

### `backend/src/config/settings.py`

Valida variables de entorno y carga DataBase.env mediante pydantic-settings.

SHA-256: `5c6d90fc3cb475df2286d69c8b2e47b501a9ba429c6c77942cd9b62376bf330e`.

**Imports directos:**

```python
from pydantic_settings import BaseSettings, SettingsConfigDict
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `Settings` | 3?13 | `class Settings` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    PROJECT_NAME: str = "Fixify API"
    VERSION: str = "1.0.0"
    ENVIRONMENT: str = "development" 
    
    # Variables obligatorias
    DATABASE_URL: str 
    SECRET_KEY: str 

    # Aquí le decimos a Pydantic que el archivo se llama exactamente "DataBase.env"
    model_config = SettingsConfigDict(env_file="DataBase.env", env_file_encoding="utf-8", extra="ignore")

settings = Settings()
```

### `backend/src/database/__init__.py`

Marcador de paquete sin l?gica propia.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

### `backend/src/database/conexion.py`

Engine, f?brica de sesiones, Base y generador get_db.

SHA-256: `1ea2c612c1f1ab3131645c9e0ae52c863fae6e05f8d477921c4f901832e05ddd`.

**Imports directos:**

```python
from sqlalchemy import create_engine
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from src.config.settings import settings
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `get_db` | 15?20 | `get_db()` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
from sqlalchemy import create_engine
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from src.config.settings import settings  # Importamos tu nuevo gestor central

# Creamos el motor usando la variable validada por Pydantic
engine = create_engine(settings.DATABASE_URL)

# Fábrica de sesiones para interactuar con la BD
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

# Base para los modelos de SQLAlchemy
Base = declarative_base()

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()
```

### `backend/src/database/domain_models.py`

Seis entidades futuras de mantenimiento; solo persistencia declarativa.

SHA-256: `f41407c7b4a9855890c3b576cffda08af7cc12b6e8bfe748df9ed30708db4956`.

**Imports directos:**

```python
from sqlalchemy import Column, Integer, String, Text, DateTime, ForeignKey, func, text
from src.database.conexion import Base
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `Material` | 6?12 | `class Material` | Material: descripción, existencias y mínimo de inventario. |
| `ServiceOrder` | 15?25 | `class ServiceOrder` | Orden: folio, propietario, equipo, técnico, estado y fechas. |
| `Diagnosis` | 28?34 | `class Diagnosis` | Diagnóstico asociado a una orden y su fecha de registro. |
| `RepairActivity` | 37?44 | `class RepairActivity` | Actividad y solución documentadas durante una reparación. |
| `MaterialConsumption` | 47?53 | `class MaterialConsumption` | Cantidad de un material utilizada en una orden. |
| `StateHistory` | 56?64 | `class StateHistory` | Cambio de estado: orden, usuario, valores anterior/nuevo y fecha. |

**Código de referencia, no ejecutable**

```python
"""Entidades restantes del SQL local; no implementan endpoints de mantenimiento."""
from sqlalchemy import Column, Integer, String, Text, DateTime, ForeignKey, func, text
from src.database.conexion import Base


class Material(Base):
    """Material: descripción, existencias y mínimo de inventario."""
    __tablename__ = 'materiales'
    id_material = Column(Integer, primary_key=True, autoincrement=True)
    descripcion = Column(String(200), nullable=False)
    existencia_actual = Column(Integer, server_default=text('0'))
    stock_minimo = Column(Integer, server_default=text('0'))


class ServiceOrder(Base):
    """Orden: folio, propietario, equipo, técnico, estado y fechas."""
    __tablename__ = 'ordenes_servicio'
    id_orden = Column(Integer, primary_key=True, autoincrement=True)
    folio = Column(String(50), unique=True, nullable=False)
    id_equipo = Column(Integer, ForeignKey('equipos.id_equipo'), nullable=False)
    id_cliente = Column(Integer, ForeignKey('clientes.id_cliente'), nullable=False)
    id_tecnico = Column(Integer, ForeignKey('usuarios.id_usuario'))
    estado = Column(String(50), nullable=False)
    fecha_recepcion = Column(DateTime, server_default=func.current_timestamp())
    fecha_entrega = Column(DateTime)


class Diagnosis(Base):
    """Diagnóstico asociado a una orden y su fecha de registro."""
    __tablename__ = 'diagnosticos'
    id_diagnostico = Column(Integer, primary_key=True, autoincrement=True)
    id_orden = Column(Integer, ForeignKey('ordenes_servicio.id_orden'), nullable=False)
    descripcion = Column(Text, nullable=False)
    fecha_registro = Column(DateTime, server_default=func.current_timestamp())


class RepairActivity(Base):
    """Actividad y solución documentadas durante una reparación."""
    __tablename__ = 'actividades_reparacion'
    id_actividad = Column(Integer, primary_key=True, autoincrement=True)
    id_orden = Column(Integer, ForeignKey('ordenes_servicio.id_orden'), nullable=False)
    trabajo_realizado = Column(Text, nullable=False)
    solucion = Column(Text)
    fecha_registro = Column(DateTime, server_default=func.current_timestamp())


class MaterialConsumption(Base):
    """Cantidad de un material utilizada en una orden."""
    __tablename__ = 'consumos_material'
    id_consumo = Column(Integer, primary_key=True, autoincrement=True)
    id_orden = Column(Integer, ForeignKey('ordenes_servicio.id_orden'), nullable=False)
    id_material = Column(Integer, ForeignKey('materiales.id_material'), nullable=False)
    cantidad_utilizada = Column(Integer, nullable=False)


class StateHistory(Base):
    """Cambio de estado: orden, usuario, valores anterior/nuevo y fecha."""
    __tablename__ = 'historial_estados'
    id_historial = Column(Integer, primary_key=True, autoincrement=True)
    id_orden = Column(Integer, ForeignKey('ordenes_servicio.id_orden'), nullable=False)
    id_usuario = Column(Integer, ForeignKey('usuarios.id_usuario'), nullable=False)
    estado_anterior = Column(String(50))
    estado_nuevo = Column(String(50), nullable=False)
    fecha_cambio = Column(DateTime, server_default=func.current_timestamp())
```

### `backend/src/middleware/__init__.py`

Marcador de paquete sin l?gica propia.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

### `backend/src/modules/__init__.py`

Marcador de paquete sin l?gica propia.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

### `backend/src/modules/auth/dependencies.py`

Extracci?n Bearer y autorización basada en claims JWT, sin consulta de estado actual.

SHA-256: `2b2825c4a35a8d335d57e7c5cba50b8ffa9ffdd318c015b6f2cd735ab61dacd7`.

**Imports directos:**

```python
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
import jwt
from src.config.settings import settings
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `verificar_usuario_autenticado` | 8?17 | `verificar_usuario_autenticado(token: str=Depends(oauth2_scheme))` | token: Bearer JWT; valida firma/expiración sin consultar estado actual en BD. |
| `verificar_rol_admin` | 19?37 | `verificar_rol_admin(token: str=Depends(oauth2_scheme))` | token: JWT firmado; exige el rol admin almacenado al iniciar sesión. |
| `verificar_rol_tecnico_o_superior` | 38?48 | `verificar_rol_tecnico_o_superior(token: str=Depends(oauth2_scheme))` | Permite el acceso a técnicos, administradores y recepción (prácticamente cualquier usuario activo). |
| `verificar_rol_recepcion_o_admin` | 50?59 | `verificar_rol_recepcion_o_admin(token: str=Depends(oauth2_scheme))` | Exclusivo para Recepción y Administración (ideal para registrar clientes, equipos u órdenes). |

**Código de referencia, no ejecutable**

```python
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
import jwt
from src.config.settings import settings

# Le indica a FastAPI de dónde viene el token
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/login")
def verificar_usuario_autenticado(token: str = Depends(oauth2_scheme)):
    """token: Bearer JWT; valida firma/expiración sin consultar estado actual en BD."""
    try:
        payload = jwt.decode(token, settings.SECRET_KEY, algorithms=["HS256"])
        return payload
    except jwt.PyJWTError:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Token inválido o expirado"
        )

def verificar_rol_admin(token: str = Depends(oauth2_scheme)):
    """token: JWT firmado; exige el rol admin almacenado al iniciar sesión."""
    try:
        # Abrimos el gafete (token) para leer su contenido
        payload = jwt.decode(token, settings.SECRET_KEY, algorithms=["HS256"])
        rol = payload.get("role")
        
        if rol != "admin":
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail="Operación denegada. Solo para administradores."
            )
        return payload # Devuelve los datos si todo está bien
        
    except jwt.PyJWTError:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Token inválido o expirado"
        )
def verificar_rol_tecnico_o_superior(token: str = Depends(oauth2_scheme)):
    """Permite el acceso a técnicos, administradores y recepción (prácticamente cualquier usuario activo)."""
    payload = verificar_usuario_autenticado(token)
    # Como todos los roles válidos entran, basta con que esté autenticado, 
    # pero puedes restringirlo explícitamente si lo deseas:
    if payload.get("role") not in ["admin", "tecnico", "recepcion"]:
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Acceso denegado."
        )
    return payload

def verificar_rol_recepcion_o_admin(token: str = Depends(oauth2_scheme)):
    """Exclusivo para Recepción y Administración (ideal para registrar clientes, equipos u órdenes)."""
    payload = verificar_usuario_autenticado(token)
    
    if payload.get("role") not in ["admin", "recepcion"]:
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Acceso exclusivo para Recepción y Administración."
        )
    return payload
```

### `backend/src/modules/auth/router.py`

Login mediante formulario con rechazo de cuenta/rol inv?lidos y generaci?n de sesión.

SHA-256: `882f031f8bfc546e52fc97f6f1f659bc27f0e611d363a2e9de3d99a3e6df4b18`.

**Imports directos:**

```python
from fastapi import APIRouter, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordRequestForm
from sqlalchemy.orm import Session
from src.database.conexion import get_db
from src.modules.users.repository import user_repository
from src.shared.security import verify_password, create_access_token
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `login` | 12?38 | `login(form_data: OAuth2PasswordRequestForm=Depends(), db: Session=Depends(get_db))` | form_data: usuario/correo y contraseña; db: sesión para verificar cuenta y rol. |

**Código de referencia, no ejecutable**

```python
from fastapi import APIRouter, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordRequestForm
from sqlalchemy.orm import Session
from src.database.conexion import get_db
from src.modules.users.repository import user_repository
from src.shared.security import verify_password, create_access_token


router = APIRouter(prefix="/auth", tags=["Autenticación"])

@router.post("/login")
def login(form_data: OAuth2PasswordRequestForm = Depends(), db: Session = Depends(get_db)):
    """form_data: usuario/correo y contraseña; db: sesión para verificar cuenta y rol."""
    # 1. Buscar al usuario en Supabase por su nombre de usuario
    user = user_repository.get_by_username(db, username=form_data.username)
    
    # 2. Validar que el usuario exista y la contraseña sea correcta
    if not user or not verify_password(form_data.password, user.hashed_password):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Usuario o contraseña incorrectos",
            headers={"WWW-Authenticate": "Bearer"},
        )
    # Rechaza cuentas desactivadas antes de emitir una sesión.
    if not user.is_active:
        raise HTTPException(status_code=403, detail="Cuenta deshabilitada. Contacte al administrador.")
    if user.role not in ('admin', 'tecnico', 'recepcion'):
        raise HTTPException(status_code=403, detail='La cuenta no tiene un rol autorizado.')
    # 3. Si es correcto, generar el token JWT incluyendo su rol
    access_token = create_access_token(data={"sub": user.username, "role": user.role})
    
    # 4. Responder al frontend
    return {
        "access_token": access_token,
        "token_type": "bearer",
        "role": user.role,
        "message": "Ingreso exitoso"
    }
```

### `backend/src/modules/client/client.controller.py`

Scaffolding histórico: no registrado como m?dulo funcional de la aplicación.

SHA-256: `04bf77c2db02fca8d13184a5359f014bda17f905c23c9ed5a3cda8c4699b8fed`.

**Código de referencia, no ejecutable**

```python
#Controlador que se encarga de definir la ruta HTTP
# (GET, POST, PUT, QUERY, PATCH o DELETE)
# ademas de que reciben la peticion del cliente
```

### `backend/src/modules/client/client.py`

Scaffolding histórico: no registrado como m?dulo funcional de la aplicación.

SHA-256: `2562d12e3f3fde62ea424b6ae2d36c0d7ef6d9e5bd8bc250e394f0dcac272d37`.

**Código de referencia, no ejecutable**

```python
# Modelo que representa la tabla o coleccion en la base de datos
# Es decir Interfaz para deserealizar lo que se obtiene de la base de datos
```

### `backend/src/modules/client/client.service.py`

Scaffolding histórico: no registrado como m?dulo funcional de la aplicación.

SHA-256: `51eb6056a937b121c768a4cefc73a02735db88f5f68b495b5f768c4b171419a7`.

**Código de referencia, no ejecutable**

```python
# Contiene las reglas y la lógica de negocio
# de tal manera se separa el "como se procesa"
# del "como se recibe", es decir:
# Reglas: definen como funciona un proceso (validaciones antes de hacer un proceso)
# Procesar datos: toma información que envia un usuario, se analiza y se decide si se almacena o se regresa.
# Portege: Controla permisos de acceso y asegura las operaciones sensibles.
```

### `backend/src/modules/clients/model.py`

Columnas del SQL; dirección es una propiedad de compatibilidad no persistida.

SHA-256: `48c8a9db468aac3fcce863894538886cbbb0f9726e48e9126d0813056161c7f1`.

**Imports directos:**

```python
from sqlalchemy import Column, Integer, String
from src.database.conexion import Base
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `Client` | 4?19 | `class Client` | Su propósito y contrato se describen en el m?dulo. |
| `Client.direccion` | 17?19 | `direccion(self)` | Compatibilidad API: el SQL base no define dirección persistida. |

**Código de referencia, no ejecutable**

```python
from sqlalchemy import Column, Integer, String
from src.database.conexion import Base

class Client(Base):
    __tablename__ = "clientes"

    # AQUÍ ESTÁ EL CAMBIO: Le indicamos que el nombre real en Supabase es "id_cliente"
    id = Column("id_cliente", Integer, primary_key=True, index=True, autoincrement=True)
    
    # --- DATOS OBLIGATORIOS (nullable=False) ---
    nombre = Column(String(150), nullable=False)
    telefono = Column(String(20), nullable=True)
    
    # --- DATOS OPCIONALES (nullable=True) ---
    correo = Column(String(100), nullable=True)     # No todos los clientes usan correo
    @property
    def direccion(self):
        """Compatibilidad API: el SQL base no define dirección persistida."""
        return None
```

### `backend/src/modules/clients/repository.py`

Alta con chequeo de teléfono, rechazo de dirección y b?squedas parciales.

SHA-256: `65aad40395bbce6e5611127ab70523b1ca002f2eded6629fa73b96399089d88f`.

**Imports directos:**

```python
from sqlalchemy.orm import Session
from fastapi import HTTPException, status
from src.modules.clients.model import Client
from src.modules.clients.schema import ClientCreate
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `ClientRepository` | 6?37 | `class ClientRepository` | Su propósito y contrato se describen en el m?dulo. |
| `ClientRepository.create` | 7?28 | `create(self, db: Session, client_in: ClientCreate)` | Su propósito y contrato se describen en el m?dulo. |
| `ClientRepository.get_all` | 30?31 | `get_all(self, db: Session)` | Su propósito y contrato se describen en el m?dulo. |
| `ClientRepository.buscar_cliente_por_termino` | 32?37 | `buscar_cliente_por_termino(self, db: Session, termino: str)` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
from sqlalchemy.orm import Session
from fastapi import HTTPException, status
from src.modules.clients.model import Client
from src.modules.clients.schema import ClientCreate

class ClientRepository:
    def create(self, db: Session, client_in: ClientCreate):
        if client_in.direccion:
            raise HTTPException(status_code=422, detail='El SQL base no incluye dirección del cliente.')
        # 1. Validar si ya existe un cliente con el mismo teléfono
        cliente_existente = db.query(Client).filter(Client.telefono == client_in.telefono).first()
        
        if cliente_existente:
            raise HTTPException(
                status_code=status.HTTP_400_BAD_REQUEST,
                detail=f"El cliente con el número de teléfono {client_in.telefono} ya se encuentra registrado."
            )
        
        # 2. Si no existe, procedemos a crearlo normalmente
        db_client = Client(
            nombre=client_in.nombre,
            telefono=client_in.telefono,
            correo=client_in.correo
        )
        db.add(db_client)
        db.commit()
        db.refresh(db_client)
        return db_client

    def get_all(self, db: Session):
        return db.query(Client).all()
    def buscar_cliente_por_termino(self, db: Session, termino: str):
        # Permite buscar coincidencias parciales por teléfono o nombre
        return db.query(Client).filter(
            (Client.telefono.ilike(f"%{termino}%")) | 
            (Client.nombre.ilike(f"%{termino}%"))
        ).all()

client_repository = ClientRepository()
```

### `backend/src/modules/clients/router.py`

Registro/listado/b?squeda exclusivos de admin y recepción.

SHA-256: `3fe47f4b166abb1673230fa3fa725c6839aa6a1c8db1d1c35e3f40e7be91f963`.

**Imports directos:**

```python
from fastapi import APIRouter, Depends
from sqlalchemy.orm import Session
from typing import List
from src.database.conexion import get_db
from src.modules.auth.dependencies import verificar_rol_recepcion_o_admin
from src.modules.clients.schema import ClientCreate, ClientResponse
from src.modules.clients.repository import client_repository
from src.modules.clients.schema import ClientCreate, ClientResponse, ClientMessageResponse
from typing import List
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `buscar_cliente_para_equipo` | 18?24 | `buscar_cliente_para_equipo(q: str, db: Session=Depends(get_db))` | Busca y confirma la existencia del cliente por nombre o teléfono  antes de proceder al registro de su equipo. |
| `registrar_cliente` | 28?35 | `registrar_cliente(cliente: ClientCreate, db: Session=Depends(get_db))` | Su propósito y contrato se describen en el m?dulo. |
| `listar_clientes` | 38?39 | `listar_clientes(db: Session=Depends(get_db))` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
from fastapi import APIRouter, Depends
from sqlalchemy.orm import Session
from typing import List
from src.database.conexion import get_db
from src.modules.auth.dependencies import verificar_rol_recepcion_o_admin
from src.modules.clients.schema import ClientCreate, ClientResponse
from src.modules.clients.repository import client_repository
from src.modules.clients.schema import ClientCreate, ClientResponse, ClientMessageResponse
from typing import List

router = APIRouter(
    prefix="/clientes",
    tags=["Gestión de Clientes"],
    dependencies=[Depends(verificar_rol_recepcion_o_admin)] # Solo Recepción o Admin pueden entrar aquí
)

@router.get("/buscar", response_model=List[ClientResponse])
def buscar_cliente_para_equipo(q: str, db: Session = Depends(get_db)):
    """
    Busca y confirma la existencia del cliente por nombre o teléfono 
    antes de proceder al registro de su equipo.
    """
    resultados = client_repository.buscar_cliente_por_termino(db, q)
    return resultados


@router.post("/", status_code=201) # 201 Created es el estándar HTTP para registros exitosos
def registrar_cliente(cliente: ClientCreate, db: Session = Depends(get_db)):
    # El repositorio ya valida si existe y lanza el error 400 si está duplicado
    nuevo_cliente = client_repository.create(db, cliente)
    
    return {
        "mensaje": "¡Cliente registrado correctamente en el sistema!",
        "cliente": nuevo_cliente
    }

@router.get("/", response_model=List[ClientResponse])
def listar_clientes(db: Session = Depends(get_db)):
    return client_repository.get_all(db)
```

### `backend/src/modules/clients/schema.py`

Validaciones de alta y lectura ORM; dirección conservada en contrato público.

SHA-256: `c5e99e653d2d750db3370c6d917379fc6f66b9a75db1365738f7eabec231638b`.

**Imports directos:**

```python
from pydantic import BaseModel, Field, EmailStr
from typing import Optional
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `ClientCreate` | 5?12 | `class ClientCreate` | Su propósito y contrato se describen en el m?dulo. |
| `ClientResponse` | 15?23 | `class ClientResponse` | Su propósito y contrato se describen en el m?dulo. |
| `ClientResponse.Config` | 22?23 | `class Config` | Su propósito y contrato se describen en el m?dulo. |
| `ClientMessageResponse` | 24?25 | `class ClientMessageResponse` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
from pydantic import BaseModel, Field, EmailStr
from typing import Optional

# Esquema para Registrar un Cliente
class ClientCreate(BaseModel):
    # OBLIGATORIO: Usamos Field(...) y un largo mínimo
    nombre: str = Field(..., min_length=2, max_length=100, description="El nombre es obligatorio")
    telefono: str = Field(..., min_length=8, max_length=20, description="Teléfono de contacto obligatorio")
    
    # OPCIONAL: Usamos Optional[tipo] = None
    correo: Optional[EmailStr] = Field(None, description="Correo electrónico opcional pero con formato válido si se proporciona")
    direccion: Optional[str] = Field(None, max_length=250, description="Dirección opcional")

# Esquema de respuesta para el cliente
class ClientResponse(BaseModel):
    id: int
    nombre: str
    telefono: str
    correo: Optional[str] = None
    direccion: Optional[str] = None

    class Config:
        from_attributes = True
class ClientMessageResponse(BaseModel):
    mensaje: str
```

### `backend/src/modules/equipment/model.py`

Equipos físicos; aliases para ID, due?o y falla.

SHA-256: `ee6a9ea70b5bc16a198c65c50712872919548c17d4847d27ab22dafd4739c540`.

**Imports directos:**

```python
from sqlalchemy import Column, Integer, String, Text, ForeignKey
from src.database.conexion import Base
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `Equipment` | 4?16 | `class Equipment` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
from sqlalchemy import Column, Integer, String, Text, ForeignKey
from src.database.conexion import Base

class Equipment(Base):
    __tablename__ = "equipos"

    # Conservamos los atributos de la API y usamos los nombres físicos del diagrama.
    id = Column("id_equipo", Integer, primary_key=True, index=True, autoincrement=True)
    tipo = Column(String(50), nullable=True)        # Ej. Laptop, Celular, Consola
    marca = Column(String(50), nullable=True)
    modelo = Column(String(50), nullable=True)
    numero_serie = Column(String(100), nullable=True)
    problema_reportado = Column("falla_reportada", Text, nullable=True)
    
    # Relación con el cliente que creamos antes
    client_id = Column("id_cliente", Integer, ForeignKey("clientes.id_cliente"), nullable=False)
```

### `backend/src/modules/equipment/repository.py`

Valida propietario, guarda y consulta con orden por ID.

SHA-256: `d36b28d81b2793bc41035e1071355d5455d1fc4cbff03ee42f5062ce8ba32fe2`.

**Imports directos:**

```python
from sqlalchemy.orm import Session
from fastapi import HTTPException, status
from src.modules.equipment.model import Equipment
from src.modules.equipment.schema import EquipmentCreate
from src.modules.clients.model import Client
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `EquipmentRepository` | 7?35 | `class EquipmentRepository` | Su propósito y contrato se describen en el m?dulo. |
| `EquipmentRepository.get_all` | 8?10 | `get_all(self, db: Session)` | db: sesión actual; devuelve los equipos ordenados por identificador. |
| `EquipmentRepository.create` | 12?35 | `create(self, db: Session, eq_in: EquipmentCreate)` | db: sesión actual; eq_in: datos validados del equipo y su propietario. |

**Código de referencia, no ejecutable**

```python
from sqlalchemy.orm import Session
from fastapi import HTTPException, status
from src.modules.equipment.model import Equipment
from src.modules.equipment.schema import EquipmentCreate
from src.modules.clients.model import Client

class EquipmentRepository:
    def get_all(self, db: Session):
        """db: sesión actual; devuelve los equipos ordenados por identificador."""
        return db.query(Equipment).order_by(Equipment.id).all()

    def create(self, db: Session, eq_in: EquipmentCreate):
        """db: sesión actual; eq_in: datos validados del equipo y su propietario."""
        # 1. Validar que el cliente dueno realmente exista
        # Solo necesitamos el ID; no consultamos otros datos del cliente.
        client = db.query(Client.id).filter(Client.id == eq_in.client_id).first()
        if not client:
            raise HTTPException(
                status_code=status.HTTP_404_NOT_FOUND,
                detail="El cliente especificado no existe en la base de datos."
            )
        
        # 2. Registrar el equipo
        db_equipment = Equipment(
            tipo=eq_in.tipo,
            marca=eq_in.marca,
            modelo=eq_in.modelo,
            numero_serie=eq_in.numero_serie,
            problema_reportado=eq_in.problema_reportado,
            client_id=eq_in.client_id
        )
        db.add(db_equipment)
        db.commit()
        db.refresh(db_equipment)
        return db_equipment

equipment_repository = EquipmentRepository()
```

### `backend/src/modules/equipment/router.py`

POST de alta con folio y GET público de entidad para la UI.

SHA-256: `999ff820e4a3a6584544c5fd5c2908f144f31343afc86ec11597d3e1cb0addaf`.

**Imports directos:**

```python
from fastapi import APIRouter, Depends
from sqlalchemy.orm import Session
from typing import List
from src.database.conexion import get_db
from src.modules.auth.dependencies import verificar_rol_recepcion_o_admin
from src.modules.equipment.schema import EquipmentCreate, EquipmentResponse
from src.modules.equipment.repository import equipment_repository
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `listar_equipos` | 16?18 | `listar_equipos(db: Session=Depends(get_db))` | db: sesión inyectada; devuelve equipos con los campos usados por el frontend. |
| `registrar_equipo` | 21?38 | `registrar_equipo(equipo: EquipmentCreate, db: Session=Depends(get_db))` | equipo: datos validados; db: sesión inyectada para guardar el registro. |

**Código de referencia, no ejecutable**

```python
from fastapi import APIRouter, Depends
from sqlalchemy.orm import Session
from typing import List
from src.database.conexion import get_db
from src.modules.auth.dependencies import verificar_rol_recepcion_o_admin
from src.modules.equipment.schema import EquipmentCreate, EquipmentResponse
from src.modules.equipment.repository import equipment_repository

router = APIRouter(
    prefix="/equipos",
    tags=["Gestión de Equipos"],
    dependencies=[Depends(verificar_rol_recepcion_o_admin)]
)

@router.get("/", response_model=List[EquipmentResponse])
def listar_equipos(db: Session = Depends(get_db)):
    """db: sesión inyectada; devuelve equipos con los campos usados por el frontend."""
    return equipment_repository.get_all(db)

@router.post("/", status_code=201)
def registrar_equipo(equipo: EquipmentCreate, db: Session = Depends(get_db)):
    """equipo: datos validados; db: sesión inyectada para guardar el registro."""
    nuevo_equipo = equipment_repository.create(db, equipo)
    
    # Generamos un folio formateado para recepción (Ej. EQ-001, EQ-015)
    folio_visible = f"EQ-{nuevo_equipo.id:03d}"
    
    return {
        "mensaje": "¡Equipo registrado correctamente!",
        "folio": folio_visible,  # <-- Esto es lo que verá la recepcionista en grande
        "detalles": {
            "id_interno": nuevo_equipo.id,
            "tipo": nuevo_equipo.tipo,
            "marca": nuevo_equipo.marca,
            "modelo": nuevo_equipo.modelo,
            "cliente_id": nuevo_equipo.client_id
        }
    }
```

### `backend/src/modules/equipment/schema.py`

Entrada y lectura de equipo, incluida serie opcional.

SHA-256: `006fd4370be98c49434071a5d1b5bcdd6ac3e44e09a1fd82460d637a5982a313`.

**Imports directos:**

```python
from pydantic import BaseModel, Field
from typing import Optional
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `EquipmentCreate` | 4?10 | `class EquipmentCreate` | Su propósito y contrato se describen en el m?dulo. |
| `EquipmentResponse` | 12?22 | `class EquipmentResponse` | Su propósito y contrato se describen en el m?dulo. |
| `EquipmentResponse.Config` | 21?22 | `class Config` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
from pydantic import BaseModel, Field
from typing import Optional

class EquipmentCreate(BaseModel):
    tipo: str = Field(..., min_length=2, max_length=50, description="Tipo de aparato")
    marca: str = Field(..., min_length=2, max_length=50)
    modelo: str = Field(..., min_length=2, max_length=50)
    numero_serie: Optional[str] = Field(None, max_length=100)
    problema_reportado: str = Field(..., min_length=5, description="Falla descrita por el cliente")
    client_id: int = Field(..., description="ID del cliente propietario")

class EquipmentResponse(BaseModel):
    id: int
    tipo: str
    marca: str
    modelo: str
    numero_serie: Optional[str] = None
    problema_reportado: str
    client_id: int

    class Config:
        from_attributes = True
```

### `backend/src/modules/roles/model.py`

Cat?logo f?sico y conversi?n de etiquetas a códigos de dominio.

SHA-256: `63d59e894432288028832618109259a2865fdf28821b30d975d40b795e1def24`.

**Imports directos:**

```python
import unicodedata
from sqlalchemy import Column, Integer, String
from src.database.conexion import Base
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `Role` | 7?23 | `class Role` | Su propósito y contrato se describen en el m?dulo. |
| `Role.code` | 13?23 | `code(self)` | Convierte etiquetas del SQL, incluidas las de prueba, al rol de dominio. |

**Código de referencia, no ejecutable**

```python
"""Roles físicos del SQL y traducción al contrato de permisos de la API."""
import unicodedata
from sqlalchemy import Column, Integer, String
from src.database.conexion import Base


class Role(Base):
    __tablename__ = 'roles'
    id = Column('id_rol', Integer, primary_key=True, autoincrement=True)
    nombre_rol = Column(String(50), nullable=False)

    @property
    def code(self):
        """Convierte etiquetas del SQL, incluidas las de prueba, al rol de dominio."""
        name = ''.join(c for c in unicodedata.normalize('NFD', self.nombre_rol.lower())
                       if not unicodedata.combining(c))
        if name.startswith('admin'):
            return 'admin'
        if name.startswith('tecnico'):
            return 'tecnico'
        if name.startswith('recep'):
            return 'recepcion'
        return 'sin_rol'
```

### `backend/src/modules/users/model.py`

Usuarios físicos, FK a roles y aliases p?blicos; is_active es persistido.

SHA-256: `a32fc302b9cbe97485f375ed440fa94d5b0ca05d9cd0361b524823f9157c96c8`.

**Imports directos:**

```python
from sqlalchemy import Column, Integer, String, ForeignKey, Boolean, true
from sqlalchemy.orm import relationship
from src.database.conexion import Base
from src.modules.roles.model import Role
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `User` | 8?22 | `class User` | Su propósito y contrato se describen en el m?dulo. |
| `User.role` | 20?22 | `role(self)` | Código de permisos derivado de la fila roles, no de otra columna. |

**Código de referencia, no ejecutable**

```python
"""Usuario SQL: aliases conservan username/id/hash del contrato público."""
from sqlalchemy import Column, Integer, String, ForeignKey, Boolean, true
from sqlalchemy.orm import relationship
from src.database.conexion import Base
from src.modules.roles.model import Role


class User(Base):
    __tablename__ = 'usuarios'
    id = Column('id_usuario', Integer, primary_key=True, autoincrement=True)
    id_rol = Column(Integer, ForeignKey('roles.id_rol'), nullable=True)
    username = Column('nombre', String(150), nullable=False)
    correo = Column(String(100), unique=True, nullable=True)
    hashed_password = Column('contrasena_hash', String(255), nullable=False)
    # Nombre físico en Supabase; is_active conserva el contrato del frontend.
    is_active = Column('is_active', Boolean, nullable=False, default=True, server_default=true())
    rol = relationship(Role, lazy='joined')

    @property
    def role(self):
        """Código de permisos derivado de la fila roles, no de otra columna."""
        return self.rol.code if self.rol else 'sin_rol'
```

### `backend/src/modules/users/repository.py`

Alta, consulta, cambios de rol y activaci?n persistente.

SHA-256: `5c6bbf54da3932c16a308274b8c380803444c75e522debbaa5ecec0a56c57d26`.

**Imports directos:**

```python
from fastapi import HTTPException
from sqlalchemy.orm import Session
from src.modules.users.model import User
from src.modules.roles.model import Role
from src.modules.users.schema import UserCreate
from src.shared.security import get_password_hash
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `UserRepository` | 9?62 | `class UserRepository` | Persistencia de usuarios y catálogo de roles; cada cambio confirma la sesión. |
| `UserRepository.get_by_username` | 11?13 | `get_by_username(self, db: Session, username: str)` | Busca por nombre de cuenta; también permite ingresar con correo. |
| `UserRepository.get_all` | 15?17 | `get_all(self, db: Session)` | db: sesión actual; devuelve cuentas ordenadas por ID. |
| `UserRepository.get_role` | 19?24 | `get_role(self, db: Session, code: str)` | Localiza la etiqueta SQL que representa el rol de dominio. |
| `UserRepository.create` | 26?35 | `create(self, db: Session, user_in: UserCreate)` | user_in: credenciales y rol; guarda hash y FK, nunca contraseña plana. |
| `UserRepository.habilitar_usuario` | 37?44 | `habilitar_usuario(self, db: Session, user_id: int)` | user_id: cuenta destino; guarda is_active=True sin cambiar su rol. |
| `UserRepository.inhabilitar_usuario` | 46?53 | `inhabilitar_usuario(self, db: Session, user_id: int)` | user_id: cuenta destino; bloquea nuevos logins sin eliminar el usuario. |
| `UserRepository.actualizar_rol` | 55?62 | `actualizar_rol(self, db: Session, user_id: int, nuevo_rol: str)` | user_id: cuenta; nuevo_rol: código que se resuelve a una fila roles. |

**Código de referencia, no ejecutable**

```python
from fastapi import HTTPException
from sqlalchemy.orm import Session
from src.modules.users.model import User
from src.modules.roles.model import Role
from src.modules.users.schema import UserCreate
from src.shared.security import get_password_hash


class UserRepository:
    """Persistencia de usuarios y catálogo de roles; cada cambio confirma la sesión."""
    def get_by_username(self, db: Session, username: str):
        """Busca por nombre de cuenta; también permite ingresar con correo."""
        return db.query(User).filter((User.username == username) | (User.correo == username)).first()

    def get_all(self, db: Session):
        """db: sesión actual; devuelve cuentas ordenadas por ID."""
        return db.query(User).order_by(User.id).all()

    def get_role(self, db: Session, code: str):
        """Localiza la etiqueta SQL que representa el rol de dominio."""
        role = next((role for role in db.query(Role).order_by(Role.id).all() if role.code == code), None)
        if role is None:
            raise HTTPException(status_code=409, detail='El rol solicitado no existe en el catálogo roles.')
        return role

    def create(self, db: Session, user_in: UserCreate):
        """user_in: credenciales y rol; guarda hash y FK, nunca contraseña plana."""
        if self.get_by_username(db, user_in.username):
            raise HTTPException(status_code=409, detail='El usuario ya existe.')
        db_user = User(username=user_in.username, hashed_password=get_password_hash(user_in.password),
                       rol=self.get_role(db, user_in.role))
        db.add(db_user)
        db.commit()
        db.refresh(db_user)
        return db_user

    def habilitar_usuario(self, db: Session, user_id: int):
        """user_id: cuenta destino; guarda is_active=True sin cambiar su rol."""
        user = db.get(User, user_id)
        if user:
            user.is_active = True
            db.commit()
            db.refresh(user)
        return user

    def inhabilitar_usuario(self, db: Session, user_id: int):
        """user_id: cuenta destino; bloquea nuevos logins sin eliminar el usuario."""
        user = db.get(User, user_id)
        if user:
            user.is_active = False
            db.commit()
            db.refresh(user)
        return user

    def actualizar_rol(self, db: Session, user_id: int, nuevo_rol: str):
        """user_id: cuenta; nuevo_rol: código que se resuelve a una fila roles."""
        user = db.get(User, user_id)
        if user:
            user.rol = self.get_role(db, nuevo_rol)
            db.commit()
            db.refresh(user)
        return user


user_repository = UserRepository()
```

### `backend/src/modules/users/router.py`

Operaciones administrativas; salida p?blica protegida por UserResponse.

SHA-256: `4aaf44ce7cca279192095d731c89fe73875bde6d39c2a2872de00ddc8b9f5263`.

**Imports directos:**

```python
from fastapi import APIRouter, Depends
from sqlalchemy.orm import Session
from src.database.conexion import get_db
from src.modules.auth.dependencies import verificar_rol_admin
from src.modules.users.repository import user_repository
from src.modules.users.schema import UserCreate, UserUpdateRole, UserResponse
from fastapi import HTTPException
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `habilitar_cuenta` | 18?26 | `habilitar_cuenta(user_id: int, db: Session=Depends(get_db))` | user_id: cuenta a activar; db: sesión inyectada; 404 si no existe. |
| `inhabilitar_cuenta` | 28?35 | `inhabilitar_cuenta(user_id: int, db: Session=Depends(get_db))` | user_id: cuenta a desactivar; conserva el registro y su rol. |
| `actualizar_rol` | 37?44 | `actualizar_rol(user_id: int, nuevo_rol: UserUpdateRole, db: Session=Depends(get_db))` | user_id: cuenta; nuevo_rol: código de rol validado; db: sesión actual. |
| `crear_usuario` | 46?50 | `crear_usuario(usuario: UserCreate, db: Session=Depends(get_db))` | usuario: datos de alta; db: sesión; devuelve el nombre confirmado. |
| `obtener_usuarios` | 53?56 | `obtener_usuarios(db: Session=Depends(get_db))` | db: sesión actual; UserResponse limita la salida a los campos públicos. |

**Código de referencia, no ejecutable**

```python
from fastapi import APIRouter, Depends
from sqlalchemy.orm import Session
from src.database.conexion import get_db
from src.modules.auth.dependencies import verificar_rol_admin
from src.modules.users.repository import user_repository
# Esquemas públicos: validan entradas y excluyen hashes de las respuestas.
from src.modules.users.schema import UserCreate, UserUpdateRole, UserResponse
from fastapi import HTTPException


# Al poner la dependencia aquí, TODOS los endpoints de este archivo exigen ser Admin
router = APIRouter(
    prefix="/usuarios",
    tags=["Gestión de Usuarios"],
    dependencies=[Depends(verificar_rol_admin)] 
)
@router.patch("/{user_id}/habilitar")
def habilitar_cuenta(user_id: int, db: Session = Depends(get_db)):
    """user_id: cuenta a activar; db: sesión inyectada; 404 si no existe."""
    # Ejecutamos la función del repositorio
    usuario = user_repository.habilitar_usuario(db, user_id)
    
    if not usuario:
        raise HTTPException(status_code=404, detail="Usuario no encontrado")
        
    return {"mensaje": f"La cuenta de {usuario.username} ha sido habilitada para acceder al sistema."}
@router.patch("/{user_id}/inhabilitar")
def inhabilitar_cuenta(user_id: int, db: Session = Depends(get_db)):
    """user_id: cuenta a desactivar; conserva el registro y su rol."""
    # Ejecutamos la función del repositorio para bloquear el acceso
    usuario = user_repository.inhabilitar_usuario(db, user_id)
    if not usuario:
        raise HTTPException(status_code=404, detail="Usuario no encontrado")
        
    return {"mensaje": f"La cuenta de {usuario.username} ha sido inhabilitada. Su acceso fue bloqueado pero la información se conservó."}
@router.patch("/{user_id}/rol")
def actualizar_rol(user_id: int, nuevo_rol: UserUpdateRole, db: Session = Depends(get_db)):
    """user_id: cuenta; nuevo_rol: código de rol validado; db: sesión actual."""
    # Ejecutamos la función del repositorio para actualizar el rol
    usuario = user_repository.actualizar_rol(db, user_id, nuevo_rol.role)
    if not usuario:
        raise HTTPException(status_code=404, detail="Usuario no encontrado")
        
    return {"mensaje": f"El rol de {usuario.username} ha sido actualizado a {usuario.role}."}
@router.post("/")
def crear_usuario(usuario: UserCreate, db: Session = Depends(get_db)):
    """usuario: datos de alta; db: sesión; devuelve el nombre confirmado."""
    # Solo un administrador con token válido puede llegar a ejecutar esto
    nuevo_usuario = user_repository.create(db, usuario)
    return {"mensaje": "Usuario creado con éxito", "usuario": nuevo_usuario.username}

@router.get("/", response_model=list[UserResponse])
def obtener_usuarios(db: Session = Depends(get_db)):
    """db: sesión actual; UserResponse limita la salida a los campos públicos."""
    # Solo un administrador puede ver la lista de empleados
    return user_repository.get_all(db)
```

### `backend/src/modules/users/schema.py`

Esquemas de alta, cambio de rol y lectura p?blica sin contraseña/hash.

SHA-256: `d5562ad6fdf5f5860c5c0796f0a20166e04f64b0e5cf25536eefcd4989ddb4f8`.

**Imports directos:**

```python
from pydantic import BaseModel, Field
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `UserCreate` | 3?19 | `class UserCreate` | Su propósito y contrato se describen en el m?dulo. |
| `UserUpdateRole` | 21?26 | `class UserUpdateRole` | Su propósito y contrato se describen en el m?dulo. |
| `UserResponse` | 28?35 | `class UserResponse` | Su propósito y contrato se describen en el m?dulo. |
| `UserResponse.Config` | 34?35 | `class Config` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
from pydantic import BaseModel, Field

class UserCreate(BaseModel):
    username: str = Field(
        ..., 
        min_length=4, 
        max_length=50, 
        description="El nombre de usuario es obligatorio y debe tener al menos 4 caracteres"
    )
    password: str = Field(
        ..., 
        min_length=6, 
        description="La contraseña es obligatoria y debe tener al menos 6 caracteres"
    )
    role: str = Field(
        default="tecnico", 
        pattern="^(admin|tecnico|recepcion)$", 
        description="Rol del usuario: admin, tecnico o recepcion"
    )

class UserUpdateRole(BaseModel):
    role: str = Field(
        ..., 
        pattern="^(admin|tecnico|recepcion)$",
        description="Nuevo rol a asignar"
    )

class UserResponse(BaseModel):
    id: int
    username: str
    role: str
    is_active: bool

    class Config:
        from_attributes = True
```

### `backend/src/services/__init__.py`

Marcador de paquete sin l?gica propia.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

### `backend/src/shared/__init__.py`

Marcador de paquete sin l?gica propia.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

### `backend/src/shared/base_repository.py`

Repositorio gen?rico preparado; no usado por los repositorios activos.

SHA-256: `422452f3866dbf260398e2906e807a1cc533a4b0841e8bc6d0f813eb3d774153`.

**Imports directos:**

```python
from typing import Generic, Type, TypeVar, List, Optional
from sqlalchemy.orm import Session
from src.database.conexion import Base
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `BaseRepository` | 10?40 | `class BaseRepository` | Su propósito y contrato se describen en el m?dulo. |
| `BaseRepository.__init__` | 11?16 | `__init__(self, model: Type[ModelType])` | Repositorio base con operaciones CRUD reutilizables. :दारा model: El modelo de SQLAlchemy (la tabla de la BD) |
| `BaseRepository.get_by_id` | 18?20 | `get_by_id(self, db: Session, id: int)` | Obtiene un registro por su ID (Deserialización desde BD) |
| `BaseRepository.get_multi` | 22?24 | `get_multi(self, db: Session, skip: int=0, limit: int=100)` | Obtiene una lista paginada de registros |
| `BaseRepository.create` | 26?33 | `create(self, db: Session, obj_in: CreateSchemaType)` | Crea un nuevo registro a partir de un esquema de Pydantic |
| `BaseRepository.remove` | 35?40 | `remove(self, db: Session, id: int)` | Elimina un registro por su ID |

**Código de referencia, no ejecutable**

```python
from typing import Generic, Type, TypeVar, List, Optional
from sqlalchemy.orm import Session
from src.database.conexion import Base

# Definimos tipos genéricos para los modelos de SQLAlchemy y los esquemas de Pydantic
ModelType = TypeVar("ModelType", bound=Base)
CreateSchemaType = TypeVar("CreateSchemaType")
UpdateSchemaType = TypeVar("UpdateSchemaType")

class BaseRepository(Generic[ModelType, CreateSchemaType, UpdateSchemaType]):
    def __init__(self, model: Type[ModelType]):
        """
        Repositorio base con operaciones CRUD reutilizables.
        :दारा model: El modelo de SQLAlchemy (la tabla de la BD)
        """
        self.model = model

    def get_by_id(self, db: Session, id: int) -> Optional[ModelType]:
        """Obtiene un registro por su ID (Deserialización desde BD)"""
        return db.query(self.model).filter(self.model.id == id).first()

    def get_multi(self, db: Session, skip: int = 0, limit: int = 100) -> List[ModelType]:
        """Obtiene una lista paginada de registros"""
        return db.query(self.model).offset(skip).limit(limit).all()

    def create(self, db: Session, obj_in: CreateSchemaType) -> ModelType:
        """Crea un nuevo registro a partir de un esquema de Pydantic"""
        obj_in_data = obj_in.model_dump() # Convierte el esquema Pydantic a diccionario
        db_obj = self.model(**obj_in_data)
        db.add(db_obj)
        db.commit()
        db.refresh(db_obj)
        return db_obj

    def remove(self, db: Session, id: int) -> ModelType:
        """Elimina un registro por su ID"""
        obj = db.query(self.model).get(id)
        db.delete(obj)
        db.commit()
        return obj
```

### `backend/src/shared/base_schema.py`

Esquema reutilizable para lectura ORM; los esquemas activos no heredan de ?l.

SHA-256: `c7b13e7e4d9310387b46eacd3fef63333830e5c1dba44b33003bff3e6cd457d2`.

**Imports directos:**

```python
from pydantic import BaseModel, ConfigDict
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `BaseSchema` | 3?8 | `class BaseSchema` | Esquema base para Pydantic.  Configurado para permitir la lectura de modelos de SQLAlchemy automáticamente. |

**Código de referencia, no ejecutable**

```python
from pydantic import BaseModel, ConfigDict

class BaseSchema(BaseModel):
    """
    Esquema base para Pydantic. 
    Configurado para permitir la lectura de modelos de SQLAlchemy automáticamente.
    """
    model_config = ConfigDict(from_attributes=True)
```

### `backend/src/shared/security.py`

bcrypt directo y JWT HS256 de ocho horas.

SHA-256: `1fbd5a4d9ade5c65ea69214d31db4dac1264f969c36b68c280998b3db6ccc696`.

**Imports directos:**

```python
from datetime import datetime, timedelta
import bcrypt
import jwt
from src.config.settings import settings
from fastapi import HTTPException
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `verify_password` | 10?15 | `verify_password(plain_password: str, hashed_password: str)` | plain_password: contraseña recibida; hashed_password: hash bcrypt almacenado. |
| `get_password_hash` | 17?21 | `get_password_hash(password: str)` | password: contraseña a convertir en hash; máximo 72 bytes en UTF-8. |
| `create_access_token` | 23?31 | `create_access_token(data: dict)` | data: claims de usuario/rol; firma un JWT HS256 con vigencia de ocho horas. |

**Código de referencia, no ejecutable**

```python
from datetime import datetime, timedelta
import bcrypt
import jwt
from src.config.settings import settings
from fastapi import HTTPException

# bcrypt genera hashes unidireccionales; las contraseñas no se cifran de forma reversible.


def verify_password(plain_password: str, hashed_password: str) -> bool:
    """plain_password: contraseña recibida; hashed_password: hash bcrypt almacenado."""
    try:
        return bcrypt.checkpw(plain_password.encode('utf-8'), hashed_password.encode('utf-8'))
    except ValueError:
        return False

def get_password_hash(password: str) -> str:
    """password: contraseña a convertir en hash; máximo 72 bytes en UTF-8."""
    if len(password.encode('utf-8')) > 72:
        raise HTTPException(status_code=422, detail='La contraseña excede el límite de 72 bytes de bcrypt.')
    return bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')

def create_access_token(data: dict) -> str:
    """data: claims de usuario/rol; firma un JWT HS256 con vigencia de ocho horas."""
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(hours=8)
    to_encode.update({"exp": expire})
    
    # SECRET_KEY se carga desde DataBase.env mediante Settings.
    encoded_jwt = jwt.encode(to_encode, settings.SECRET_KEY, algorithm="HS256")
    return encoded_jwt
```

### `backend/tests/test_equipment.py`

Contrato de equipos con tablas independientes y transporte ASGI de pruebas.

SHA-256: `838845420bfc05f779e69090472d9ee19ee572e4761566a7c38b3206d9890532`.

**Imports directos:**

```python
import os
from pathlib import Path
import sys
import unittest
from datetime import datetime, timedelta, timezone
import jwt
import httpx
from fastapi import FastAPI
from fastapi.testclient import TestClient
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
from sqlalchemy.pool import StaticPool
from src.config.settings import settings
from src.database.conexion import get_db
from src.modules.equipment.model import Equipment
from src.modules.equipment.router import router
from services import Services
from services.api_client import ApiClient
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `EquipmentTests` | 30?131 | `class EquipmentTests` | Comprueba alta/listado, serialización, columnas y permisos de equipos. |
| `EquipmentTests.setUp` | 33?60 | `setUp(self)` | Crea tablas independientes del ORM con los nombres del diagrama. |
| `EquipmentTests.setUp.override_db` | 49?52 | `override_db()` | Entrega una sesión de prueba y la cierra al terminar la petición. |
| `EquipmentTests.tearDown` | 62?65 | `tearDown(self)` | Libera servidor y base temporal. |
| `EquipmentTests.auth_headers` | 68?73 | `auth_headers(role)` | role: rol simulado; firma un JWT exclusivamente de pruebas. |
| `EquipmentTests.test_create_and_list_against_diagram_columns` | 75?87 | `test_create_and_list_against_diagram_columns(self)` | Alta y recarga conservan contrato API sin exigir renombrar la BD. |
| `EquipmentTests.test_missing_client_and_invalid_payload` | 89?95 | `test_missing_client_and_invalid_payload(self)` | Rechaza cliente inexistente y datos inválidos sin insertar equipos. |
| `EquipmentTests.test_optional_serial_and_order` | 97?104 | `test_optional_serial_and_order(self)` | Serie ausente retorna null; listado tiene orden estable por ID. |
| `EquipmentTests.test_permissions` | 106?113 | `test_permissions(self)` | Conserva acceso admin/recepción y rechaza técnico o falta de JWT. |
| `EquipmentTests.test_frontend_service_consumes_real_router` | 115?131 | `test_frontend_service_consumes_real_router(self)` | httpx del frontend usa el router real en memoria mediante ASGITransport. |
| `EquipmentTests.test_frontend_service_consumes_real_router.flow` | 119?129 | `flow()` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
"""Prueba equipos con nombres físicos del diagrama, sin Supabase ni main.py."""
import os
from pathlib import Path
import sys
import unittest
from datetime import datetime, timedelta, timezone

# Configuración aislada: las pruebas no necesitan DataBase.env ni conexión remota.
os.environ['DATABASE_URL'] = 'sqlite://'
os.environ['SECRET_KEY'] = 'fixify-equipment-tests-only-secret-key-32-chars'
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / 'frontend/src'))

import jwt
import httpx
from fastapi import FastAPI
from fastapi.testclient import TestClient
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
from sqlalchemy.pool import StaticPool

from src.config.settings import settings
from src.database.conexion import get_db
from src.modules.equipment.model import Equipment
from src.modules.equipment.router import router
from services import Services
from services.api_client import ApiClient


class EquipmentTests(unittest.TestCase):
    """Comprueba alta/listado, serialización, columnas y permisos de equipos."""

    def setUp(self):
        """Crea tablas independientes del ORM con los nombres del diagrama."""
        self.engine = create_engine('sqlite://', connect_args={'check_same_thread': False}, poolclass=StaticPool)
        self.sessions = sessionmaker(bind=self.engine)
        with self.engine.begin() as connection:
            connection.execute(text('PRAGMA foreign_keys=ON'))
            connection.execute(text('''CREATE TABLE clientes (
                id_cliente INTEGER PRIMARY KEY, nombre VARCHAR(150),
                telefono VARCHAR(20), correo VARCHAR(100))'''))
            connection.execute(text('''CREATE TABLE equipos (
                id_equipo INTEGER PRIMARY KEY AUTOINCREMENT,
                id_cliente INTEGER NOT NULL REFERENCES clientes(id_cliente),
                tipo VARCHAR(50), marca VARCHAR(50), modelo VARCHAR(50),
                numero_serie VARCHAR(100), falla_reportada TEXT)'''))
            connection.execute(text("INSERT INTO clientes VALUES (7, 'Cliente prueba', '3312345678', NULL)"))

        def override_db():
            """Entrega una sesión de prueba y la cierra al terminar la petición."""
            with self.sessions() as session:
                yield session

        self.app = FastAPI()
        self.app.include_router(router)
        self.app.dependency_overrides[get_db] = override_db
        self.client = TestClient(self.app)
        self.headers = self.auth_headers('recepcion')
        self.payload = {'tipo': 'Laptop', 'marca': 'Marca', 'modelo': 'Modelo',
                        'numero_serie': 'PRUEBA-001', 'problema_reportado': 'No enciende', 'client_id': 7}

    def tearDown(self):
        """Libera servidor y base temporal."""
        self.client.close()
        self.engine.dispose()

    @staticmethod
    def auth_headers(role):
        """role: rol simulado; firma un JWT exclusivamente de pruebas."""
        token = jwt.encode({'sub': 'prueba', 'role': role,
                            'exp': datetime.now(timezone.utc) + timedelta(minutes=5)},
                           settings.SECRET_KEY, algorithm='HS256')
        return {'Authorization': f'Bearer {token}'}

    def test_create_and_list_against_diagram_columns(self):
        """Alta y recarga conservan contrato API sin exigir renombrar la BD."""
        response = self.client.post('/equipos/', json=self.payload, headers=self.headers)
        self.assertEqual(response.status_code, 201, response.text)
        self.assertEqual(response.json()['folio'], 'EQ-001')
        response = self.client.get('/equipos/', headers=self.headers)
        self.assertEqual(response.status_code, 200, response.text)
        self.assertEqual(response.json(), [{'id': 1, **self.payload}])
        with self.engine.connect() as connection:
            row = connection.execute(text('SELECT id_equipo, id_cliente, falla_reportada FROM equipos')).one()
            self.assertEqual(tuple(row), (1, 7, 'No enciende'))
        fk = next(iter(Equipment.__table__.c.id_cliente.foreign_keys))
        self.assertEqual(fk.column.name, 'id_cliente')

    def test_missing_client_and_invalid_payload(self):
        """Rechaza cliente inexistente y datos inválidos sin insertar equipos."""
        response = self.client.post('/equipos/', json={**self.payload, 'client_id': 999}, headers=self.headers)
        self.assertEqual(response.status_code, 404)
        response = self.client.post('/equipos/', json={**self.payload, 'tipo': ''}, headers=self.headers)
        self.assertEqual(response.status_code, 422)
        self.assertEqual(self.client.get('/equipos/', headers=self.headers).json(), [])

    def test_optional_serial_and_order(self):
        """Serie ausente retorna null; listado tiene orden estable por ID."""
        for serial in (None, 'SEGUNDO'):
            response = self.client.post('/equipos/', json={**self.payload, 'numero_serie': serial}, headers=self.headers)
            self.assertEqual(response.status_code, 201)
        items = self.client.get('/equipos/', headers=self.headers).json()
        self.assertEqual([item['id'] for item in items], [1, 2])
        self.assertIsNone(items[0]['numero_serie'])

    def test_permissions(self):
        """Conserva acceso admin/recepción y rechaza técnico o falta de JWT."""
        for method in ('get', 'post'):
            kwargs = {'json': self.payload} if method == 'post' else {}
            request = getattr(self.client, method)
            self.assertEqual(request('/equipos/', **kwargs).status_code, 401)
            self.assertEqual(request('/equipos/', headers=self.auth_headers('tecnico'), **kwargs).status_code, 403)
            self.assertIn(request('/equipos/', headers=self.auth_headers('admin'), **kwargs).status_code, (200, 201))

    def test_frontend_service_consumes_real_router(self):
        """httpx del frontend usa el router real en memoria mediante ASGITransport."""
        import asyncio

        async def flow():
            api = ApiClient(transport=httpx.ASGITransport(app=self.app))
            api.set_token(self.headers['Authorization'].removeprefix('Bearer '))
            services = Services(api)
            try:
                created = await services.equipment.crear(**self.payload)
                items = await services.equipment.listar()
                self.assertEqual(created.folio, 'EQ-001')
                self.assertEqual([item.model_dump() for item in items], [{'id': 1, **self.payload}])
            finally:
                await api.aclose()

        asyncio.run(flow())


if __name__ == '__main__':
    unittest.main()
```

### `backend/tests/test_schema_contract.py`

Comparaci?n con SQL y regresiones de autenticación/estado.

SHA-256: `f887b06b0adb3023dccd5fa7cdb21837a112901e165bdcb03898e0c13d94e77c`.

**Imports directos:**

```python
import os
from pathlib import Path
import re
import sys
import unittest
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from sqlalchemy.pool import StaticPool
from fastapi import FastAPI
from fastapi.testclient import TestClient
from src.database.conexion import Base, get_db
from src.database import domain_models
from src.modules.clients.model import Client
from src.modules.equipment.model import Equipment
from src.modules.users.model import User
from src.modules.roles.model import Role
from src.modules.users.repository import user_repository
from src.modules.users.schema import UserCreate
from src.modules.auth.router import router
from src.shared.security import verify_password
```

| S?mbolo | L?neas | Firma / argumentos | Prop?sito |
|---|---|---|---|
| `SchemaTests` | 28?90 | `class SchemaTests` | Su propósito y contrato se describen en el m?dulo. |
| `SchemaTests.test_actual_account_state_column` | 29?32 | `test_actual_account_state_column(self)` | Regresión del 500: PostgreSQL confirmó usuarios.is_active, no is_activate. |
| `SchemaTests.test_all_sql_columns_and_foreign_keys` | 34?51 | `test_all_sql_columns_and_foreign_keys(self)` | El SQL local, no el ORM, define las columnas físicas esperadas. |
| `SchemaTests.test_create_and_authenticate_sql_user` | 53?90 | `test_create_and_authenticate_sql_user(self)` | Cuenta con nombre, correo, hash e id_rol autentica con el contrato actual. |
| `SchemaTests.test_create_and_authenticate_sql_user.override` | 67?69 | `override()` | Su propósito y contrato se describen en el m?dulo. |

**Código de referencia, no ejecutable**

```python
"""Comprueba columnas SQL y autenticación con usuarios/roles, sin conexión remota."""
import os
from pathlib import Path
import re
import sys
import unittest

os.environ['DATABASE_URL'] = 'sqlite://'
os.environ['SECRET_KEY'] = 'schema-tests-secret-only-32-characters'
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from sqlalchemy.pool import StaticPool
from fastapi import FastAPI
from fastapi.testclient import TestClient
from src.database.conexion import Base, get_db
from src.database import domain_models
from src.modules.clients.model import Client
from src.modules.equipment.model import Equipment
from src.modules.users.model import User
from src.modules.roles.model import Role
from src.modules.users.repository import user_repository
from src.modules.users.schema import UserCreate
from src.modules.auth.router import router
from src.shared.security import verify_password


class SchemaTests(unittest.TestCase):
    def test_actual_account_state_column(self):
        """Regresión del 500: PostgreSQL confirmó usuarios.is_active, no is_activate."""
        self.assertEqual(User.is_active.property.columns[0].name, 'is_active')
        self.assertNotIn('is_activate', User.__table__.columns)

    def test_all_sql_columns_and_foreign_keys(self):
        """El SQL local, no el ORM, define las columnas físicas esperadas."""
        sql = (Path(__file__).resolve().parents[2] / 'database/scripts/BASE DE DATOS FIXIFY.sql').read_text(encoding='utf-8')
        definitions = re.findall(r'CREATE TABLE (\w+)\s*\((.*?)\);', sql, re.S)
        self.assertEqual(set(Base.metadata.tables), {name for name, _ in definitions})
        for name, body in definitions:
            expected = set(re.findall(r'^\s*(\w+)\s+(?:SERIAL|VARCHAR|INT|TEXT|TIMESTAMP|BOOLEAN)\b', body, re.M))
            table = Base.metadata.tables[name]
            self.assertEqual(set(table.columns.keys()), expected, name)
            for column_name, definition in re.findall(r'^\s*(\w+)\s+((?:SERIAL|VARCHAR|INT|TEXT|TIMESTAMP|BOOLEAN)[^,\n]*)', body, re.M):
                column = table.columns[column_name]
                self.assertEqual(column.nullable, not ('NOT NULL' in definition or 'PRIMARY KEY' in definition), f'{name}.{column_name}')
                length = re.search(r'VARCHAR\((\d+)\)', definition)
                if length:
                    self.assertEqual(column.type.length, int(length.group(1)))
            expected_fk = set(re.findall(r'FOREIGN KEY \((\w+)\) REFERENCES (\w+)\((\w+)\)', body))
            actual_fk = {(fk.parent.name, fk.column.table.name, fk.column.name) for fk in table.foreign_keys}
            self.assertEqual(actual_fk, expected_fk, name)

    def test_create_and_authenticate_sql_user(self):
        """Cuenta con nombre, correo, hash e id_rol autentica con el contrato actual."""
        engine = create_engine('sqlite://', connect_args={'check_same_thread': False}, poolclass=StaticPool)
        Base.metadata.create_all(engine)
        with Session(engine) as db:
            db.add(Role(nombre_rol='Recepcionista (prueba)'))
            db.commit()
            user = user_repository.create(db, UserCreate(username='prueba_test', password='Prueba-segura-123', role='recepcion'))
            self.assertEqual(user.role, 'recepcion')
            self.assertTrue(user.is_active)
            identifier = user.id
            self.assertTrue(verify_password('Prueba-segura-123', user.hashed_password))
            self.assertFalse(verify_password('incorrecta', user.hashed_password))

        def override():
            with Session(engine) as db:
                yield db

        app = FastAPI()
        app.include_router(router)
        app.dependency_overrides[get_db] = override
        with TestClient(app) as client:
            response = client.post('/auth/login', data={'username': 'prueba_test', 'password': 'Prueba-segura-123'})
            self.assertEqual(response.status_code, 200, response.text)
            self.assertEqual(response.json()['role'], 'recepcion')
            self.assertEqual(client.post('/auth/login', data={'username': 'prueba_test', 'password': 'incorrecta'}).status_code, 401)
            with Session(engine) as db:
                user_repository.inhabilitar_usuario(db, identifier)
            self.assertEqual(client.post('/auth/login', data={'username': 'prueba_test', 'password': 'Prueba-segura-123'}).status_code, 403)
            with Session(engine) as db:
                user = db.get(User, identifier)
                self.assertFalse(user.is_active)
                self.assertEqual(user.role, 'recepcion')
                user_repository.habilitar_usuario(db, identifier)
                self.assertIsNone(user_repository.habilitar_usuario(db, 99999))
                self.assertIsNone(user_repository.inhabilitar_usuario(db, 99999))
            self.assertEqual(client.post('/auth/login', data={'username': 'prueba_test', 'password': 'Prueba-segura-123'}).status_code, 200)
        engine.dispose()
```

## 10. Inspección local segura

Esta celda revisa sintaxis y calcula huellas de archivos Python, sin importar la aplicación. Una validación de sintaxis no es una prueba de persistencia ni de conectividad.

In [ ]:
from pathlib import Path
import ast
import hashlib

raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'backend/main.py').is_file()), None)
if raiz is None:
    raise RuntimeError('Abra el notebook dentro del repositorio Fixify.')
archivos = [p for p in sorted((raiz / 'backend').rglob('*.py'))
            if '__pycache__' not in p.parts]
for archivo in archivos:
    ast.parse(archivo.read_text(encoding='utf-8-sig'))
print('Archivos Python con sintaxis v?lida:', len(archivos))
print('Huella de main.py:', hashlib.sha256((raiz / 'backend/main.py').read_bytes()).hexdigest()[:12])
print('No se import? la API ni se conect? a Supabase.')

## Referencias del proyecto

- Código vigente: backend/main.py, backend/src/, backend/scripts/ y backend/tests/.
- SQL local: database/scripts/BASE DE DATOS FIXIFY.sql.
- Migración: database/migrations/001_equipment_reported_fault.sql.
- [Detalle de alineación](ALINEACION_BASE_DATOS.md).
- [Documentación frontend v2](Documentacion_frontend_v2.ipynb).
- [Plan](Plan_Proyecto.ipynb).
- Diagrama aportado y traceback de PostgreSQL que confirmó usuarios.is_active.

Los requisitos de los PDF son referencias de producto; los modelos de mantenimiento no se presentan como implementación funcional completa de esos requisitos.